In [23]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Tuple, Dict, Any, List

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from ExperimentUtil import (
    run_dict,
    open_var_from_files,
    get_comp_dir,
    find_nc_files,
    collect_variable_list,
    flatten_variable_dict,
    open_compvar,
    set_monthly_time_to_midmonth_cftime
)

In [39]:
# -----------------------------
# Config
# -----------------------------
@dataclass
class NaoDailyProjectConfig:
    eof_base_nc: str

    # NAO definition domain
    lat_bounds: Tuple[float, float] = (20.0, 80.0)
    lon_bounds: Tuple[float, float] = (-90.0, 40.0)  # [-180, 180] convention
    months_ndjfm: Tuple[int, ...] = (11, 12, 1, 2, 3)

    # Standardization stats taken from base file nao_pc1 over this period (NDJFM only)
    base_start: str = "1985-01-01"
    base_end: str = "2014-12-31"

    # Input variable detection
    var_candidates: Tuple[str, ...] = ("PSL", "psl", "SLP", "slp", "msl", "MSL", "PMSL", "pmsl")

    # Daily anomaly method
    # "monthly_climo_from_base": subtract base_climatology(month,lat,lon) from EOF base
    daily_anom_method: str = "monthly_climo_from_base"

    # Windowing for lead metrics
    window_days: int = 14
    max_lead_days: int = 84  # e.g. 6 windows (=12 weeks)

    # Grid handling
    interp_to_eof_grid: bool = True
    round_coords_decimals: int = 6


# -----------------------------
# I/O helper
# -----------------------------
def to_daily_mean(
    da: xr.DataArray,
    *,
    time_name: str = "time",
) -> xr.DataArray:
    """
    Convert sub-daily (e.g., 6-hourly) to daily means.
    Assumes time coordinate is decodeable and monotonic.
    Output time stamps are daily at 00:00.
    """
    if time_name not in da.coords:
        # try common alternatives
        for cand in ("time", "Time", "date", "datetime"):
            if cand in da.coords:
                da = da.rename({cand: "time"})
                time_name = "time"
                break
    if time_name not in da.coords:
        raise KeyError(f"to_daily_mean: cannot find time coord. coords={list(da.coords)}")

    # sort to make resample safe
    try:
        da = da.sortby(time_name)
    except Exception:
        pass

    # daily mean; skipna for safety
    daily = da.resample({time_name: "1D"}).mean(time_name, skipna=True)

    # drop all-empty days if any exist
    daily = daily.dropna(time_name, how="all")
    return daily
    
def open_daily_psl_from_dir(
    root: str,
    sub: str,
    var: str = "PSL",
    time_slice=None,
    scale: float = 1.0,
    *,
    mfdataset_kwargs: Optional[Dict[str, Any]] = None,
) -> xr.DataArray:
    """
    Convenience reader for daily PSL stored as many NetCDF files under root/sub.
    """
    d = Path(root) / sub
    files = sorted(d.glob("*.nc"))
    if len(files) == 0:
        raise FileNotFoundError(f"No NetCDF files found under: {d}")

    kwargs = dict(combine="by_coords")
    if mfdataset_kwargs:
        kwargs.update(mfdataset_kwargs)

    ds = xr.open_mfdataset(files, **kwargs)

    if var not in ds:
        raise KeyError(f"Variable '{var}' not found in {d}. Available: {list(ds.data_vars)}")

    da = ds[var] * scale
    if time_slice is not None:
        da = da.sel(time=slice(time_slice[0], time_slice[1]))
    return da


# -----------------------------
# Projector + diagnostics
# -----------------------------
class NaoDailyProjector:
    """
    Project daily-mean SLP onto a fixed NAO EOF1 pattern (from EOF base file).

    Outputs:
      - nao_daily_raw: weighted projection (Pa units, consistent with base build)
      - nao_daily_std: standardized using base nao_pc1 NDJFM stats over base_start/base_end

    Also provides:
      - biweekly means from init_date
      - temporal correlation (single-init) within lead windows
      - one-call saving of all single-init diagnostics
    """

    def __init__(self, cfg: NaoDailyProjectConfig):
        self.cfg = cfg
        self.base = xr.open_dataset(cfg.eof_base_nc)

        for v in ("nao_eof1", "base_climatology", "nao_pc1"):
            if v not in self.base:
                raise KeyError(f"EOF base missing '{v}'. Present: {list(self.base.data_vars)}")

        self.eof = self._canonicalize_lonlat(self.base["nao_eof1"])
        self.clim = self._canonicalize_lonlat(self.base["base_climatology"])

        # Reference stats for standardization (NDJFM only over base period)
        ref_pc = self.base["nao_pc1"].sel(time=slice(cfg.base_start, cfg.base_end))
        ref_pc = ref_pc.sel(time=ref_pc["time"].dt.month.isin(list(cfg.months_ndjfm)))
        self.ref_mean = float(ref_pc.mean("time").values)
        self.ref_std = float(ref_pc.std("time").values)
        if not np.isfinite(self.ref_std) or self.ref_std == 0.0:
            raise ValueError("Reference PC std invalid. Check base period or base file.")

    def close(self):
        try:
            self.base.close()
        except Exception:
            pass

    # -------------------------
    # Core projection (from Dataset file)
    # -------------------------
    def project_daily_file(
        self,
        in_file: str | Path,
        out_file: Optional[str | Path] = None,
        var_name: Optional[str] = None,
        dataset_name: str = "DATA",
        select_ndjfm_only: bool = False,
    ) -> xr.Dataset:
        """
        Project daily SLP to daily NAO index (raw + standardized).
        """
        in_file = Path(in_file)
        ds = xr.open_dataset(in_file)

        da = self._select_var(ds, var_name)
        out = self.project_daily_da(
            da,
            dataset_name=dataset_name,
            select_ndjfm_only=select_ndjfm_only,
        )

        if out_file is not None:
            out_file = Path(out_file)
            out_file.parent.mkdir(parents=True, exist_ok=True)
            out.to_netcdf(out_file)

        ds.close()
        return out

    # -------------------------
    # Core projection (from DataArray in memory)
    # -------------------------
    def project_daily_da(
        self,
        da: xr.DataArray,
        *,
        dataset_name: str = "DATA",
        select_ndjfm_only: bool = False,
        var_name_for_attrs: Optional[str] = None,
    ) -> xr.Dataset:
        """
        Same as project_daily_file, but takes a DataArray directly.
        This is the preferred path if you already opened daily PSL via open_mfdataset.

        Expects da has coords time/lat/lon OR common variants (latitude/longitude).
        """
        da = self._rename_common_coords(da)

        for c in ("time", "lat", "lon"):
            if c not in da.coords:
                raise KeyError(f"Input DataArray missing coord '{c}'. Coords={list(da.coords)}")

        da = self._to_lon180(da)
        da = self._subset_domain(da)

        if select_ndjfm_only:
            da = da.sel(time=da["time"].dt.month.isin(list(self.cfg.months_ndjfm)))

        if self.cfg.interp_to_eof_grid:
            da = da.interp(lat=self.eof["lat"], lon=self.eof["lon"])

        da = self._canonicalize_lonlat(da)

        eof = self.eof.sel(lat=da["lat"], lon=da["lon"])
        clim = self.clim.sel(lat=da["lat"], lon=da["lon"])

        if self.cfg.daily_anom_method == "monthly_climo_from_base":
            anom = da.groupby("time.month") - clim
        else:
            raise ValueError(f"Unknown daily_anom_method: {self.cfg.daily_anom_method}")

        w_sqrt = np.sqrt(np.cos(np.deg2rad(anom["lat"]))).clip(min=0.0)
        W = w_sqrt.broadcast_like(anom.isel(time=0))

        nao_daily_raw = ((anom * W) * eof).sum(("lat", "lon"))
        nao_daily_std = (nao_daily_raw - self.ref_mean) / self.ref_std

        # attributes
        src_name = var_name_for_attrs or getattr(da, "name", None) or "PSL"

        out = xr.Dataset(
            data_vars=dict(
                nao_daily_raw=nao_daily_raw.astype(np.float64),
                nao_daily_std=nao_daily_std.astype(np.float64),
            ),
            coords=dict(time=nao_daily_raw["time"]),
            attrs=dict(
                nao_definition="EOF1 of NDJFM SLP anomalies (20–80N, 90W–40E); daily index via projection",
                eof_base_file=str(self.cfg.eof_base_nc),
                dataset_name=str(dataset_name),
                input_var=str(src_name),
                anomaly_method=str(self.cfg.daily_anom_method),
                standardization=(
                    "nao_daily_std = (nao_daily_raw - ref_mean) / ref_std; "
                    "ref stats from base nao_pc1 NDJFM over base period"
                ),
                ref_mean=float(self.ref_mean),
                ref_std=float(self.ref_std),
            ),
        )
        return out

    # -------------------------
    # Derived diagnostics
    # -------------------------
    def biweekly_means_from_init(
        self,
        nao_daily: xr.DataArray,
        init_date: str,
        max_lead_days: Optional[int] = None,
        window_days: Optional[int] = None,
        require_full_window: bool = True,
        day1_is_init_plus_1: bool = True,
    ) -> xr.DataArray:
        """
        Convert daily NAO to consecutive window means starting at init_date.

        Default lead definition:
          lead day 1 = init_date + 1 day  (day1_is_init_plus_1=True)
        """
        window_days = int(window_days or self.cfg.window_days)
        max_lead_days = int(max_lead_days or self.cfg.max_lead_days)

        t0 = np.datetime64(init_date)
        start = t0 + np.timedelta64(1, "D") if day1_is_init_plus_1 else t0
        end = t0 + np.timedelta64(max_lead_days, "D")

        series = nao_daily.sel(time=slice(start, end))

        nwin = max_lead_days // window_days
        vals = []
        labels = []
        for i in range(nwin):
            w_start = start + np.timedelta64(i * window_days, "D")
            w_end = w_start + np.timedelta64(window_days - 1, "D")
            w = series.sel(time=slice(w_start, w_end))

            if require_full_window and int(w.sizes.get("time", 0)) < window_days:
                vals.append(np.nan)
            else:
                vals.append(float(w.mean("time").values))

            labels.append(f"Weeks {2*i+1}-{2*i+2}")

        # IMPORTANT: netCDF attrs cannot store boolean dtype; use int 0/1
        return xr.DataArray(
            np.array(vals, dtype=np.float64),
            dims=("window",),
            coords={"window": np.array(labels, dtype=object)},
            name=f"{nao_daily.name}_biweekly",
            attrs=dict(
                init_date=str(init_date),
                window_days=int(window_days),
                max_lead_days=int(max_lead_days),
                require_full_window=int(1 if require_full_window else 0),
                day1_is_init_plus_1=int(1 if day1_is_init_plus_1 else 0),
            ),
        )

    def temporal_corr_in_windows_single_init(
        self,
        nao_fcst_daily: xr.DataArray,
        nao_obs_daily: xr.DataArray,
        init_date: str,
        windows: Tuple[Tuple[int, int], ...] = ((1, 14), (15, 28), (29, 42)),
        ndjfm_only: bool = True,
        day1_is_init_plus_1: bool = True,
        min_samples: int = 5,
    ) -> xr.DataArray:
        """
        Single-init diagnostic:
          Temporal correlation of daily NAO within each lead window (correlation across days).
    
        Robust to cftime vs numpy datetime, timestamp hour offsets, and calendar differences
        by matching on YYYY-MM-DD strings instead of time objects.
        """
    
        fc = nao_fcst_daily.sortby("time")
        ob = nao_obs_daily.sortby("time")
    
        # Optional NDJFM filter (safe for both cftime + np.datetime64)
        if ndjfm_only:
            months = list(self.cfg.months_ndjfm)
            fc = fc.sel(time=fc["time"].dt.month.isin(months))
            ob = ob.sel(time=ob["time"].dt.month.isin(months))
    
        # Build YYYY-MM-DD keys (strip any hour/minute differences)
        fc_dates = np.array([str(t)[:10] for t in fc["time"].values], dtype=object)
        ob_dates = np.array([str(t)[:10] for t in ob["time"].values], dtype=object)
    
        if fc_dates.size == 0:
            raise ValueError("Forecast NAO series has zero length after filtering.")
        if ob_dates.size == 0:
            raise ValueError("Obs NAO series has zero length after filtering.")
    
        # Find init index in forecast axis
        hits = np.where(fc_dates == init_date)[0]
        if hits.size == 0:
            raise KeyError(
                f"init_date {init_date} not found in forecast time axis. "
                f"First={fc_dates[0]}, Last={fc_dates[-1]}, n={fc_dates.size}"
            )
        i0 = int(hits[0])
    
        # Map obs date -> obs index (if duplicates exist, keep first)
        ob_map: Dict[str, int] = {}
        for j, d in enumerate(ob_dates):
            if d not in ob_map:
                ob_map[d] = j
    
        # For each forecast time, get matching obs index (or -1 if missing)
        ob_j_for_fc = np.array([ob_map.get(d, -1) for d in fc_dates], dtype=int)
    
        # Lead day based on forecast index (calendar-safe)
        idx = np.arange(fc_dates.size, dtype=int)
        lead = (idx - i0) if day1_is_init_plus_1 else (idx - i0 + 1)
    
        vals = []
        labels = []
    
        fc_vals = fc.values
        ob_vals = ob.values
    
        for (d0, d1) in windows:
            d0 = int(d0)
            d1 = int(d1)
    
            # in lead window AND obs exists for that day
            m = (lead >= d0) & (lead <= d1) & (ob_j_for_fc >= 0)
    
            if not np.any(m):
                r = np.nan
                n_used = 0
            else:
                x = fc_vals[m]
                y = ob_vals[ob_j_for_fc[m]]
                good = np.isfinite(x) & np.isfinite(y)
                n_used = int(good.sum())
    
                if n_used >= int(min_samples):
                    r = float(np.corrcoef(x[good], y[good])[0, 1])
                else:
                    r = np.nan
    
            # label
            if (d1 - d0 + 1) == 14 and (d0 - 1) % 14 == 0:
                i = (d0 - 1) // 14
                lab = f"Weeks {2*i+1}-{2*i+2}"
            else:
                lab = f"Days {d0}-{d1}"
    
            vals.append(r)
            labels.append(lab)
    
        return xr.DataArray(
            np.array(vals, dtype=np.float64),
            dims=("window",),
            coords={"window": np.array(labels, dtype=object)},
            name="nao_temporal_corr",
            attrs=dict(
                definition="Temporal correlation of daily NAO within lead window (single initialization)",
                init_date=str(init_date),
                ndjfm_only=int(1 if ndjfm_only else 0),
                day1_is_init_plus_1=int(1 if day1_is_init_plus_1 else 0),
                min_samples=int(min_samples),
                windows=str(windows),
                note="Matched forecast/obs by YYYY-MM-DD strings (robust to calendar/time-type mismatch).",
            ),
        )

    def save_all_single_init_diagnostics(
        self,
        *,
        out_dir: str | Path,
        init_date: str,
        obs_psl: xr.DataArray,
        fcst_psl: xr.DataArray,
        obs_name: str = "ERA5",
        fcst_name: str = "FCST",
        windows: Optional[Tuple[Tuple[int, int], ...]] = None,
        ndjfm_only_for_acc: bool = False,
        day1_is_init_plus_1: bool = True,
        bundle_one_file: bool = True,
    ) -> Dict[str, str]:
        """
        One-call driver for single initialization:

        Saves:
          - {obs_name}.nao_daily.nc
          - {fcst_name}.nao_daily.nc
          - {obs_name}.nao_biweekly_from_init.nc
          - {fcst_name}.nao_biweekly_from_init.nc
          - {fcst_name}_vs_{obs_name}.nao_temporal_acc_14d.nc   (ACC-like metric)
          - optionally: a bundled file with all series + acc

        Returns dict of written paths.
        """
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        if windows is None:
            # default: consecutive 14-day windows up to max_lead_days
            nwin = int(self.cfg.max_lead_days // self.cfg.window_days)
            windows = tuple(
                (1 + i * self.cfg.window_days, (i + 1) * self.cfg.window_days) for i in range(nwin)
            )

        written: Dict[str, str] = {}

        # 1) Project daily NAO (obs + forecast)
        ds_obs = self.project_daily_da(obs_psl, dataset_name=obs_name, var_name_for_attrs="PSL")
        ds_fc  = self.project_daily_da(fcst_psl, dataset_name=fcst_name, var_name_for_attrs="PSL")

        f_obs_daily = out_dir / f"{obs_name}.nao_daily.nc"
        f_fc_daily  = out_dir / f"{fcst_name}.nao_daily.nc"
        ds_obs.to_netcdf(f_obs_daily)
        ds_fc.to_netcdf(f_fc_daily)
        written["obs_daily"] = str(f_obs_daily)
        written["fcst_daily"] = str(f_fc_daily)

        # 2) Biweekly means from init
        biw_obs = self.biweekly_means_from_init(
            ds_obs["nao_daily_std"],
            init_date=init_date,
            day1_is_init_plus_1=day1_is_init_plus_1,
        )
        biw_fc = self.biweekly_means_from_init(
            ds_fc["nao_daily_std"],
            init_date=init_date,
            day1_is_init_plus_1=day1_is_init_plus_1,
        )

        f_obs_biw = out_dir / f"{obs_name}.nao_biweekly_from_init.nc"
        f_fc_biw  = out_dir / f"{fcst_name}.nao_biweekly_from_init.nc"
        xr.Dataset({"nao_biweekly_std": biw_obs}).to_netcdf(f_obs_biw)
        xr.Dataset({"nao_biweekly_std": biw_fc}).to_netcdf(f_fc_biw)
        written["obs_biweekly"] = str(f_obs_biw)
        written["fcst_biweekly"] = str(f_fc_biw)

        # 3) Temporal ACC-like metric in windows (single init)
        acc = self.temporal_corr_in_windows_single_init(
            ds_fc["nao_daily_std"],
            ds_obs["nao_daily_std"],
            init_date=init_date,
            windows=windows,
            ndjfm_only=ndjfm_only_for_acc,
            day1_is_init_plus_1=day1_is_init_plus_1,
        )
        f_acc = out_dir / f"{fcst_name}_vs_{obs_name}.nao_temporal_acc_14d.nc"
        xr.Dataset({"nao_acc": acc}).to_netcdf(f_acc)
        written["temporal_acc"] = str(f_acc)

        # 4) Optional bundle
        if bundle_one_file:
            bundle = xr.Dataset(
                data_vars=dict(
                    obs_nao_daily_std=ds_obs["nao_daily_std"],
                    fcst_nao_daily_std=ds_fc["nao_daily_std"],
                    obs_nao_biweekly_std=biw_obs,
                    fcst_nao_biweekly_std=biw_fc,
                    nao_temporal_acc=acc,
                ),
                attrs=dict(
                    init_date=str(init_date),
                    obs_name=str(obs_name),
                    fcst_name=str(fcst_name),
                    definition="Single-init NAO diagnostics: projected daily index + biweekly means + temporal ACC in lead windows",
                    eof_base_file=str(self.cfg.eof_base_nc),
                ),
            )
            f_bundle = out_dir / f"{fcst_name}_vs_{obs_name}.nao_single_init_bundle.nc"
            bundle.to_netcdf(f_bundle)
            written["bundle"] = str(f_bundle)

        return written

    # -------------------------
    # Multi-start ACC helper (kept for completeness)
    # -------------------------
    def biweekly_means_from_init(
        self,
        nao_daily: xr.DataArray,
        init_date: str,
        max_lead_days: Optional[int] = None,
        window_days: Optional[int] = None,
        require_full_window: bool = True,
        day1_is_init_plus_1: bool = True,
    ) -> xr.DataArray:
        """
        Calendar-safe window means for a single initialization.
        Avoids np.datetime64 <-> cftime comparisons by slicing via integer lead-day index.
    
        Assumes nao_daily is daily means (1 value per day) and ordered in time.
        """
    
        window_days = int(window_days or self.cfg.window_days)
        max_lead_days = int(max_lead_days or self.cfg.max_lead_days)
    
        # Sort time to ensure deterministic indexing
        nao_daily = nao_daily.sortby("time")
    
        # Find init index by string match (works for cftime + datetime64)
        time_str = np.array([str(t)[:10] for t in nao_daily["time"].values], dtype=object)  # "YYYY-MM-DD"
        hits = np.where(time_str == init_date)[0]
        if hits.size == 0:
            # give a helpful error
            raise KeyError(
                f"init_date {init_date} not found in nao_daily time axis. "
                f"First={time_str[0]}, Last={time_str[-1]}, n={time_str.size}"
            )
        i0 = int(hits[0])
    
        # Lead day 1 definition
        start_i = i0 + (1 if day1_is_init_plus_1 else 0)
        end_i = start_i + max_lead_days  # exclusive
    
        series = nao_daily.isel(time=slice(start_i, min(end_i, nao_daily.sizes["time"])))
    
        nwin = max_lead_days // window_days
        vals = []
        labels = []
    
        for i in range(nwin):
            wi0 = i * window_days
            wi1 = wi0 + window_days  # exclusive
            w = series.isel(time=slice(wi0, wi1))
    
            if require_full_window and int(w.sizes.get("time", 0)) < window_days:
                vals.append(np.nan)
            else:
                vals.append(float(w.mean("time").values))
    
            labels.append(f"Weeks {2*i+1}-{2*i+2}")
    
        return xr.DataArray(
            np.array(vals, dtype=np.float64),
            dims=("window",),
            coords={"window": np.array(labels, dtype=object)},
            name=f"{nao_daily.name}_biweekly",
            attrs=dict(
                init_date=str(init_date),
                window_days=int(window_days),
                max_lead_days=int(max_lead_days),
                require_full_window=int(1 if require_full_window else 0),
                day1_is_init_plus_1=int(1 if day1_is_init_plus_1 else 0),
            ),
        )

    # -------------------------
    # Helpers
    # -------------------------
    def _select_var(self, ds: xr.Dataset, var_name: Optional[str]) -> xr.DataArray:
        if var_name is not None:
            if var_name not in ds:
                raise KeyError(f"Requested var '{var_name}' not found. Available: {list(ds.data_vars)}")
            return ds[var_name]
        for v in self.cfg.var_candidates:
            if v in ds.data_vars:
                return ds[v]
        raise KeyError(f"Could not find SLP var among {self.cfg.var_candidates}. Available: {list(ds.data_vars)}")

    def _rename_common_coords(self, da: xr.DataArray) -> xr.DataArray:
        coord_map = {}
        for k in list(da.coords):
            lk = k.lower()
            if lk == "latitude" and "lat" not in da.coords:
                coord_map[k] = "lat"
            elif lk == "longitude" and "lon" not in da.coords:
                coord_map[k] = "lon"
            elif lk == "date" and "time" not in da.coords:
                coord_map[k] = "time"
        return da.rename(coord_map) if coord_map else da

    def _to_lon180(self, da: xr.DataArray) -> xr.DataArray:
        if "lon" not in da.coords:
            return da
        lon = da["lon"].values
        if lon.ndim != 1:
            return da
        if np.nanmax(lon) > 180.0 + 1e-6:
            lon_new = ((lon + 180.0) % 360.0) - 180.0
            da = da.assign_coords(lon=lon_new).sortby("lon")
        return da

    def _subset_domain(self, da: xr.DataArray) -> xr.DataArray:
        lat0, lat1 = self.cfg.lat_bounds
        lon0, lon1 = self.cfg.lon_bounds

        if float(da["lat"].values[0]) < float(da["lat"].values[-1]):
            da = da.sel(lat=slice(lat0, lat1))
        else:
            da = da.sel(lat=slice(lat1, lat0))

        da = da.sel(lon=slice(lon0, lon1))
        return da

    def _canonicalize_lonlat(self, da: xr.DataArray) -> xr.DataArray:
        d = int(self.cfg.round_coords_decimals)
        da = self._to_lon180(da)
        if "lon" in da.coords:
            da = da.assign_coords(lon=np.round(da["lon"].values.astype(np.float64), d)).sortby("lon")
        if "lat" in da.coords:
            da = da.assign_coords(lat=np.round(da["lat"].values.astype(np.float64), d)).sortby("lat")
        return da


In [42]:
if __name__ == "__main__":
    diag_dir = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper"
    out_dir = Path(diag_dir) / "figure_data" / "nao" / "hindcast_index"
    out_dir.mkdir(parents=True, exist_ok=True)

    exp_list = ["CTRL", "DARTOP", "ERA5OP", "ERA5DL", "ERA5BL", "DARTBL", "DARTBA", "DARTAE"]

    # hindcast period
    period = "2012-2013"
    start, end = "2012-01-01", "2013-12-31"
    time_slice = (start, end)

    # init date for lead windows
    init_date = "2012-01-01"

    # NAO EOF base (your saved file)
    eof_base = Path(diag_dir) / "figure_data" / "nao" / "nao_eof_base_NOAA20C_1985-2014_NDJFM_20-80N_90W-40E.nc"

    # -----------------------
    # Reference data (ERA5 daily PSL)
    # -----------------------
    ref_dict = {
        "PSL": {
            "name": "ERA5",
            "path": "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings/ERA5",
            "sub": "post/atm/180x360_aave/ts/daily/1yr",
            "period": period,
            "var": "PSL",
            "scale": 1.0,
        },
    }

    # -----------------------
    # Model experiment fields (template for ExperimentUtil.open_compvar)
    # -----------------------
    # IMPORTANT: open_compvar expects run dict keys like "atm" (not "sub")
    # Your ExperimentUtil maps component="EAM" -> run_key="atm"
    cfgs = {
        "PSL": {
            "run_key": "native",
            "atm": "post/atm/180x360_aave/ts/6hourly/1yr",  # <-- MUST be 'atm'
            "var": "PSL",
            "is_cmip6": False,
            "scale": 1.0,
        },
    }

    # -----------------------
    # NAO projector config
    # -----------------------
    nao_cfg = NaoDailyProjectConfig(
        eof_base_nc=str(eof_base),
        window_days=14,
        max_lead_days=84,
        interp_to_eof_grid=True,
        daily_anom_method="monthly_climo_from_base",
    )
    proj = NaoDailyProjector(nao_cfg)

    # -----------------------
    # Helper: project + save daily + biweekly
    # -----------------------
    def project_and_save(name: str, da_psl: xr.DataArray) -> xr.Dataset:
        # time slice (daily)
        da_psl = da_psl.sel(time=slice(time_slice[0], time_slice[1]))

        # project directly from DataArray (NO temp PSL file)
        ds_nao = proj.project_daily_da(
            da_psl,
            dataset_name=name,
            select_ndjfm_only=False,
            var_name_for_attrs="PSL",
        )

        # 1) save daily projected NAO series
        daily_nc = out_dir / f"{name}.nao_daily.nc"
        ds_nao.to_netcdf(daily_nc)

        # 2) save biweekly means from init
        biw = proj.biweekly_means_from_init(
            ds_nao["nao_daily_std"],
            init_date=init_date,
            max_lead_days=nao_cfg.max_lead_days,
            window_days=nao_cfg.window_days,
            require_full_window=True,
            day1_is_init_plus_1=True,
        )
        biw_nc = out_dir / f"{name}.nao_biweekly_from_init.nc"
        xr.Dataset({"nao_biweekly_std": biw}).to_netcdf(biw_nc)

        print(f"[{name}] wrote: {daily_nc.name}, {biw_nc.name}")
        return ds_nao
        
    # -----------------------
    # 1) ERA5 reference NAO (daily + biweekly)
    # -----------------------
    da_ref = open_daily_psl_from_dir(
        root=ref_dict["PSL"]["path"],
        sub=ref_dict["PSL"]["sub"],
        var=ref_dict["PSL"]["var"],
        time_slice=time_slice,
        scale=ref_dict["PSL"]["scale"],
    )
    ds_era5 = project_and_save("ERA5", da_ref)

    # keep ERA5 daily NAO std for ACC
    nao_era5_daily = ds_era5["nao_daily_std"]
    
    # -----------------------
    # 2) Model experiments: daily+biweekly + temporal ACC vs ERA5
    # -----------------------
    for exp in exp_list:
        cfg = cfgs["PSL"].copy()
        cfg["name"] = run_dict[exp]['name']
        cfg["period"] = period
        run_base_dir = f"/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings"  # <-- replace with your existing run_dict lookup
        cfg["path"] = run_base_dir
 
        # NOTE: component must be "EAM" per your ExperimentUtil Known=['EAM','EAM_CMIP6',...]
        da_fcst_6h = open_compvar(cfg, component="EAM", var="PSL")
        
        # convert 6-hourly -> daily mean
        da_fcst = to_daily_mean(da_fcst_6h)
        dts = da_fcst["time"].values[:5]
        print(exp, "daily sample times:", [str(t) for t in dts])

        ds_fcst = project_and_save(exp, da_fcst)
        
        # --- temporal “ACC” in 14-day windows (single init) ---
        # default windows: (1–14, 15–28, ..., 71–84)
        nwin = nao_cfg.max_lead_days // nao_cfg.window_days
        windows = tuple((1 + i * nao_cfg.window_days, (i + 1) * nao_cfg.window_days) for i in range(nwin))

        acc = proj.temporal_corr_in_windows_single_init(
            nao_fcst_daily=ds_fcst["nao_daily_std"],
            nao_obs_daily=nao_era5_daily,
            init_date=init_date,
            windows=windows,
            ndjfm_only=False,          # keep full lead window evolution
            day1_is_init_plus_1=True,
        )

        acc_nc = out_dir / f"{exp}_vs_ERA5.nao_temporal_acc_14d.nc"
        xr.Dataset({"nao_acc": acc}).to_netcdf(acc_nc)
        
        def to_time_str_axis(da: xr.DataArray, name: str = "time_str") -> xr.DataArray:
            da = da.sortby("time")
            t = np.array([str(x)[:10] for x in da["time"].values], dtype=object)
            out = da.copy()
            out = out.assign_coords({name: ("time", t)})
            out = out.swap_dims({"time": name}).drop_vars("time")
            out = out.rename({name: "time"})
            return out
        # --- SAFE bundle (time as YYYY-MM-DD strings) ---
        fc_ts = to_time_str_axis(ds_fcst["nao_daily_std"])
        ob_ts = to_time_str_axis(nao_era5_daily)
        
        # Align on string dates (now legal)
        fc_ts, ob_ts = xr.align(fc_ts, ob_ts, join="outer")
        
        bundle_nc = out_dir / f"{exp}_vs_ERA5.nao_bundle_single_init.nc"
        xr.Dataset(
            data_vars=dict(
                nao_fcst_daily_std=fc_ts,
                nao_era5_daily_std=ob_ts,
                nao_fcst_biweekly_std=proj.biweekly_means_from_init(ds_fcst["nao_daily_std"], init_date=init_date),
                nao_era5_biweekly_std=proj.biweekly_means_from_init(nao_era5_daily, init_date=init_date),
                nao_temporal_acc=acc,
            ),
            attrs=dict(
                init_date=init_date,
                fcst_name=exp,
                obs_name="ERA5",
                eof_base_file=str(eof_base),
                definition="Single-init NAO diagnostics: daily projection + biweekly means + temporal ACC in lead windows",
                time_note="Daily series stored on string time axis YYYY-MM-DD to avoid calendar conflicts (cftime vs pandas).",
            ),
        ).to_netcdf(bundle_nc)

        print(f"[{exp}] wrote: {acc_nc.name}, {bundle_nc.name}")

    proj.close()
    print(f"All done. Outputs in: {out_dir}")
    

/tmp/ipykernel_2176924/372199690.py:90: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, **kwargs)
/tmp/ipykernel_2176924/372199690.py:90: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, **kwargs)
/tmp/ipykernel_2176924/372199690.py:90: FutureWarning: In a 

[ERA5] wrote: ERA5.nao_daily.nc, ERA5.nao_biweekly_from_init.nc


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/nao_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")


CTRL daily sample times: ['2012-01-01 00:00:00', '2012-01-02 00:00:00', '2012-01-03 00:00:00', '2012-01-04 00:00:00', '2012-01-05 00:00:00']
[CTRL] wrote: CTRL.nao_daily.nc, CTRL.nao_biweekly_from_init.nc
[CTRL] wrote: CTRL_vs_ERA5.nao_temporal_acc_14d.nc, CTRL_vs_ERA5.nao_bundle_single_init.nc


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/nao_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")


...

/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/nao_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")


DARTBA daily sample times: ['2012-01-01 00:00:00', '2012-01-02 00:00:00', '2012-01-03 00:00:00', '2012-01-04 00:00:00', '2012-01-05 00:00:00']
[DARTBA] wrote: DARTBA.nao_daily.nc, DARTBA.nao_biweekly_from_init.nc
[DARTBA] wrote: DARTBA_vs_ERA5.nao_temporal_acc_14d.nc, DARTBA_vs_ERA5.nao_bundle_single_init.nc


/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/nao_analysis/ExperimentUtil.py:143: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(files, combine="by_coords")


DARTAE daily sample times: ['2012-01-01 00:00:00', '2012-01-02 00:00:00', '2012-01-03 00:00:00', '2012-01-04 00:00:00', '2012-01-05 00:00:00']
[DARTAE] wrote: DARTAE.nao_daily.nc, DARTAE.nao_biweekly_from_init.nc


MergeError: conflicting values for variable 'month' on objects to be combined. You can skip this check by specifying compat='override'.